In [5]:
# processing Morphologist morphometry info csv 
# read in the rest of the info and combine with morphometry info
# add asymmetry index
# write the above data to csv file if needed

In [3]:
import numpy as np
import pandas as pd

In [7]:
###############################  Composing measures combing Atril, Biosca and Cemoi  ##############################

In [5]:
###################################  Prepare measures  ##################################
# Load distance measures
curProject = 'amputee'
curRegion = 'CSSyl'  # CSSyl or CSpreCS
curRoot = 'C'  # 'C' or 'D'

curRegionLeft = 'S.C._left' # 'S.C.sylvian._left'
curRegionRight = 'S.C._right' # 'S.C.sylvian._right'
path_left = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\form_measure\{curRegionLeft}.csv'
path_right = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\form_measure\{curRegionRight}.csv'

try:
    left = pd.read_csv(path_left, index_col=0, header=0, sep=';')
    right = pd.read_csv(path_right, index_col=0, header=0, sep=';')    
except FileNotFoundError as e:
    print(f"Error: {e}")

# adding side info to the index and remove the 'side' column
left.index = 'L' + left.index 
right.index = 'flip-R' + right.index
del left['side']
del right['side']
combined_measures = pd.concat([left,right],axis=0)

In [7]:
print(len(combined_measures))
print(combined_measures.index)
print(combined_measures.columns)

130
Index(['LMA03_struct_nf', 'LMA05_struct_nf', 'LMA08_struct_nf',
       'LMA12_struct_nf', 'LMA14_struct_nf', 'LMA26_struct_nf',
       'LMA28_struct_nf', 'LMA30_struct_nf', 'LMA34_struct_nf', 'LPA01_struct',
       ...
       'flip-RPC15_struct', 'flip-RPC16_struct', 'flip-RPC17_struct',
       'flip-RPC18_struct', 'flip-RPC19_struct', 'flip-RPC20_struct',
       'flip-RPC21_struct', 'flip-RPC22_struct', 'flip-RPC23_struct',
       'flip-RPC24_struct'],
      dtype='object', name='subject', length=130)
Index(['label', 'surface_talairach', 'surface_native', 'maxdepth_talairach',
       'maxdepth_native', 'meandepth_talairach', 'meandepth_native',
       'hull_junction_length_talairach', 'hull_junction_length_native',
       'GM_thickness', 'opening'],
      dtype='object')


In [9]:
##################################  Loading existing information  #################################

curDistType = 'min'         #### !!!!!!!!!!!!!!!  CHANGE  !!!!!!!!!!!!!!  ###

################################  Read the CSV file into a DataFrame  ################################
file_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curRegion}_combined_{curDistType}_2024.csv'
#file_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curRegion}_combined_{curDistType}_2025.csv'
print(file_path)

merged_info = pd.read_csv(file_path)
print(len(merged_info))

C:\B_projWIP\proj_amputee\Analysis_2025\CSSyl_combined_min_2024.csv
130


In [11]:
merged_info.columns

Index(['subjName', 'iso1', 'iso2', 'iso3', 'UMAP1_U1', 'UMAP1_U2', 'UMAP1_U3',
       'UMAP2_U3', 'UMAP1_U4', 'UMAP2_U4', 'SubjID', 'Study', 'Hemisphere',
       'Gender', 'AgeScan', 'AgeLimbLoss', 'Group', 'AmpSide', 'DominantHand',
       'Group_num', 'Gender_function', 'AmpSide_function', 'birthyear',
       'Prosthesisusage', 'Stumpusage', 'amputatio level', 'Cos', 'Fun', 'Myo',
       'missinghand', 'intacthand', 'residualarm', 'intactarm', 'lips', 'feet',
       'missing_hem'],
      dtype='object')

In [13]:
##############################  combining the morphometry measures and all the rest  ################################
combined_measures_allInfo = pd.merge(combined_measures, merged_info, left_index=True, right_on="subjName")
# Set 'subjName' as the index
combined_measures_allInfo.set_index('subjName', inplace=True)
# Remove the column 'label'
combined_measures_allInfo = combined_measures_allInfo.drop(columns=['label'])

In [15]:
combined_measures_allInfo.columns

Index(['surface_talairach', 'surface_native', 'maxdepth_talairach',
       'maxdepth_native', 'meandepth_talairach', 'meandepth_native',
       'hull_junction_length_talairach', 'hull_junction_length_native',
       'GM_thickness', 'opening', 'iso1', 'iso2', 'iso3', 'UMAP1_U1',
       'UMAP1_U2', 'UMAP1_U3', 'UMAP2_U3', 'UMAP1_U4', 'UMAP2_U4', 'SubjID',
       'Study', 'Hemisphere', 'Gender', 'AgeScan', 'AgeLimbLoss', 'Group',
       'AmpSide', 'DominantHand', 'Group_num', 'Gender_function',
       'AmpSide_function', 'birthyear', 'Prosthesisusage', 'Stumpusage',
       'amputatio level', 'Cos', 'Fun', 'Myo', 'missinghand', 'intacthand',
       'residualarm', 'intactarm', 'lips', 'feet', 'missing_hem'],
      dtype='object')

In [17]:
#########################  Adding duration_amputation column to the final complete_info  ##########################
combined_measures_allInfo['duration_amputation'] = combined_measures_allInfo['AgeScan'] - combined_measures_allInfo['AgeLimbLoss']

merged_info = combined_measures_allInfo

In [19]:
#########################  Adding asymmetry index to the final complete_info  ##########################
left_hem = merged_info[merged_info['Hemisphere'] == 'Left']
right_hem = merged_info[merged_info['Hemisphere'] == 'Right']

# DataFrame with missing hemisphere:
missing_hem = merged_info[
    ((merged_info['Hemisphere'] == 'Left') & (merged_info['AmpSide'] == 'R')) |
    ((merged_info['Hemisphere'] == 'Right') & (merged_info['AmpSide'] == 'L'))
]
# DataFrame with using hemisphere:
using_hem = merged_info[
    ((merged_info['Hemisphere'] == 'Left') & (merged_info['AmpSide'] == 'L')) |
    ((merged_info['Hemisphere'] == 'Right') & (merged_info['AmpSide'] == 'R'))
]


In [23]:
# Columns to compute asymmetry for
asyCols = ['surface_talairach','maxdepth_talairach','meandepth_talairach',
           'hull_junction_length_talairach','GM_thickness','opening','iso1','UMAP1_U2']

# Function to compute Asymmetry Index (AI) for a subject
def compute_ai(row, col):
    subjID = row['SubjID']
    group = row['Group']
    
    if group == 'CTR':
        # Dominant vs Non-Dominant hemisphere
        #dom_hemi = row['DominantHand']
        #nondom_hemi = 'R' if dom_hemi == 'L' else 'L'
        nondom_hemi = row['DominantHand']
        dom_hemi = 'R' if nondom_hemi == 'L' else 'L'        
        df_subj = merged_info[merged_info['SubjID'] == subjID]
        dom_val = df_subj.loc[df_subj['Hemisphere'] == dom_hemi, col].values
        nondom_val = df_subj.loc[df_subj['Hemisphere'] == nondom_hemi, col].values
        
    else:  # CONG or AMP
        # Using vs Missing hemisphere
        df_subj = merged_info[merged_info['SubjID'] == subjID]
        dom_val = df_subj.loc[df_subj['Hemisphere'] == df_subj['AmpSide'].values[0], col].values
        nondom_val = df_subj.loc[df_subj['Hemisphere'] == df_subj['missing_hem'].values[0], col].values

    if len(dom_val) == 0 or len(nondom_val) == 0:
        return None
    
    return (dom_val[0] - nondom_val[0]) / (dom_val[0] + nondom_val[0])


# Apply to all asymmetry columns
for col in asyCols:
    merged_info[col + '_asy'] = merged_info.apply(lambda row: compute_ai(row, col), axis=1)


In [27]:
# Columns to compute asymmetry for
asyCols = ['surface_talairach','maxdepth_talairach','meandepth_talairach',
           'hull_junction_length_talairach','GM_thickness','opening','iso1','UMAP1_U2']

# AmpSide and missing_hem contains L/R, Hemisphere contains Left/Right
hand_map = {"L": "Left", "R": "Right"}

def compute_ai(row, col):
    subjID = row['SubjID']
    group = row['Group']
    df_subj = merged_info[merged_info['SubjID'] == subjID]

    if group == 'CTR': # Dominant vs Non-Dominant     
        dom_hemi = hand_map.get(row['DominantHand'])
        nondom_hemi = "Right" if dom_hemi == "Left" else "Left"
        dom_val = df_subj.loc[df_subj['Hemisphere'] == dom_hemi, col].values
        nondom_val = df_subj.loc[df_subj['Hemisphere'] == nondom_hemi, col].values

    else:  # CONG or AMP: Using vs Missing hemisphere
        using_hemi = hand_map.get(row['AmpSide']) # AmpSide marks USING hemisphere (ipsilateral to intact hand)
        missing_hemi = hand_map.get(row['missing_hem'])   # column existing, contralateral to AmpSide
        dom_val = df_subj.loc[df_subj['Hemisphere'] == using_hemi, col].values
        nondom_val = df_subj.loc[df_subj['Hemisphere'] == missing_hemi, col].values

    if len(dom_val) == 0 or len(nondom_val) == 0:
        return None
    
    return (dom_val[0] - nondom_val[0]) / (dom_val[0] + nondom_val[0])

# Apply to all asymmetry columns
for col in asyCols:
    merged_info[col + '_asy'] = merged_info.apply(lambda row: compute_ai(row, col), axis=1)


In [29]:
# Columns to compute asymmetry for
asyCols = ['surface_talairach','maxdepth_talairach','meandepth_talairach',
           'hull_junction_length_talairach','GM_thickness','opening','iso1','UMAP1_U2']

# AmpSide and missing_hem contains L/R, Hemisphere contains Left/Right
hand_map = {"L": "Left", "R": "Right"}

def compute_ai(row, col):
    subjID = row['SubjID']
    group = row['Group']
    df_subj = merged_info[merged_info['SubjID'] == subjID]

    if group == 'CTR': # Dominant vs Non-Dominant
        dom_hemi = hand_map.get(row['DominantHand'])
        nondom_hemi = "Right" if dom_hemi == "Left" else "Left"
        dom_val = df_subj.loc[df_subj['Hemisphere'] == dom_hemi, col].values
        nondom_val = df_subj.loc[df_subj['Hemisphere'] == nondom_hemi, col].values

    else:  # CONG or AMP: Using vs Missing hemisphere
        using_hemi = hand_map.get(row['AmpSide']) # AmpSide = Using hemisphere
        missing_hemi = hand_map.get(row['missing_hem']) # column existing, contralateral to AmpSide
        dom_val = df_subj.loc[df_subj['Hemisphere'] == using_hemi, col].values
        nondom_val = df_subj.loc[df_subj['Hemisphere'] == missing_hemi, col].values

    if len(dom_val) == 0 or len(nondom_val) == 0:
        return None
    
    denom = dom_val[0] + nondom_val[0]
    if denom == 0:
        return None
    
    return (dom_val[0] - nondom_val[0]) / denom


def compute_lr_ai(row, col):
    subjID = row['SubjID']
    df_subj = merged_info[merged_info['SubjID'] == subjID]

    left_val = df_subj.loc[df_subj['Hemisphere'] == "Left", col].values
    right_val = df_subj.loc[df_subj['Hemisphere'] == "Right", col].values

    if len(left_val) == 0 or len(right_val) == 0:
        return None
    
    denom = left_val[0] + right_val[0]
    if denom == 0:
        return None
    
    return (left_val[0] - right_val[0]) / denom


# Apply both versions for all asymmetry columns
for col in asyCols:
    merged_info[col + '_asy'] = merged_info.apply(lambda row: compute_ai(row, col), axis=1)
    merged_info[col + '_LRasy'] = merged_info.apply(lambda row: compute_lr_ai(row, col), axis=1)


In [31]:
print(merged_info.columns)

Index(['surface_talairach', 'surface_native', 'maxdepth_talairach',
       'maxdepth_native', 'meandepth_talairach', 'meandepth_native',
       'hull_junction_length_talairach', 'hull_junction_length_native',
       'GM_thickness', 'opening', 'iso1', 'iso2', 'iso3', 'UMAP1_U1',
       'UMAP1_U2', 'UMAP1_U3', 'UMAP2_U3', 'UMAP1_U4', 'UMAP2_U4', 'SubjID',
       'Study', 'Hemisphere', 'Gender', 'AgeScan', 'AgeLimbLoss', 'Group',
       'AmpSide', 'DominantHand', 'Group_num', 'Gender_function',
       'AmpSide_function', 'birthyear', 'Prosthesisusage', 'Stumpusage',
       'amputatio level', 'Cos', 'Fun', 'Myo', 'missinghand', 'intacthand',
       'residualarm', 'intactarm', 'lips', 'feet', 'missing_hem',
       'duration_amputation', 'surface_talairach_asy',
       'maxdepth_talairach_asy', 'meandepth_talairach_asy',
       'hull_junction_length_talairach_asy', 'GM_thickness_asy', 'opening_asy',
       'iso1_asy', 'UMAP1_U2_asy', 'surface_talairach_LRasy',
       'maxdepth_talairach_LRas

In [35]:
################################  Saving csv files if needed  #################################

#file_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curRegion}_withMorphometry_{curDistType}_2024.csv'
file_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curRegion}_withMorphometry_asy_{curDistType}_2024.csv'
print(file_path)

# Write the DataFrame to a CSV file
#merged_info.to_csv(file_path, index=True)

C:\B_projWIP\proj_amputee\Analysis_2025\CSSyl_withMorphometry_asy_min_2024.csv
